In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import pandas as pd
import ast


In [3]:


# -------------------------------
# Input file
# -------------------------------
input_file = "results_deterministic_approximate.csv"

# -------------------------------
# Step 1: Robust manual parsing
# -------------------------------
rows = []

with open(input_file, "r") as f:
    for line_id, line in enumerate(f):
        line = line.strip()

        if not line:
            continue

        # Split ONLY first 3 commas
        parts = line.split(",", 3)

        if len(parts) < 4:
            print(f"Skipping malformed line {line_id}: {line}")
            continue

        file = parts[0]

        try:
            objective = float(parts[1])
            time = float(parts[2])
        except ValueError:
            print(f"Skipping numeric parse error at line {line_id}")
            continue

        solutions_raw = parts[3]
        if solutions_raw.strip() == "[]":
            rows.append([file, objective, time, []])
            continue    

        #print(solutions_raw.strip()[1:100])  # Debug: print first 100 chars of solutions_raw
        # Convert string list → Python list
        try:
            solutions = [ int(x.strip()) + 1 for x in (solutions_raw.strip()[1: -1]).split(",") ]  # Remove outer quotes
        except Exception:
            print(f"Warning: failed to parse solutions at line {line_id}")
            solutions = []

        rows.append([file, objective, time, solutions])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df = pd.DataFrame(rows, columns=["file", "objective", "time", "solutions"])

# -------------------------------
# Step 3: Filename parsing
# -------------------------------
def split_filename(fname):
    fname = fname.replace(".json", "")
    parts = fname.split("_")

    if len(parts) < 4:
        return pd.Series([fname, "", "", ""])

    if len(parts) > 4:
        instance_name = "_".join(parts[:-3])
        measure = parts[-3]
        percentile = parts[-2]
        algorithm = parts[-1]
    else:
        instance_name = parts[0]
        measure = parts[1]
        percentile = parts[2]
        algorithm = parts[3]

    return pd.Series([instance_name, measure, percentile, algorithm])

df[["file_name", "measure", "percentile", "algorithm"]] = df["file"].apply(split_filename)

# -------------------------------
# Step 4: Reorder columns
# -------------------------------
df = df[
    [
        "file_name",
        "measure",
        "percentile",
        "algorithm",
        "objective",
        "time",
        "solutions",
    ]
]

# -------------------------------
# Step 5: Add useful derived data
# -------------------------------
df["solution_size"] = df["solutions"].apply(len)

# Optional: density-like metric
df["solution_density"] = df["solution_size"] / df["objective"]

# -------------------------------
# Step 6: Memory optimization
# -------------------------------
df["objective"] = df["objective"].astype("int32")
df["time"] = df["time"].astype("float32")

# -------------------------------
# Step 7: Preview & sanity checks
# -------------------------------
df_greedy = df
df_greedy



Skipping numeric parse error at line 0


,file_name,measure,percentile,algorithm,objective,time,solutions,solution_size,solution_density
0,ela,cosine,p95,alg0,7805,25.053507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7805,1.0
1,transoptas_5pso,correlation,p80,alg1,5,1.834159,"[6565, 4326, 6566, 6567, 4398]",5,1.0
2,transoptas_5pso,cosine,p10,alg0,491,165.350204,"[4098, 8196, 4104, 4106, 4107, 4108, 4109, 411...",491,1.0
3,tinytla,seuclidean,p95,alg0,7463,19.015779,"[1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 1...",7463,1.0
4,transoptas_5pso,correlation,p20,alg0,963,98.866585,"[4098, 6147, 8196, 4101, 6, 2053, 4104, 4106, ...",963,1.0
...,...,...,...,...,...,...,...,...,...
175,ela,cosine,p80,alg1,13,3.561427,"[6753, 2342, 7465, 4938, 4940, 1549, 5071, 424...",13,1.0
176,transoptas_5pso,cosine,p80,alg0,5414,45.035755,"[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 1...",5414,1.0
177,deepela_large_r1,cosine,p50,alg1,2,3.385980,"[6719, 6576]",2,1.0
178,ela,cosine,p90,alg0,7476,33.878899,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7476,1.0


In [4]:
import pandas as pd
import ast

# -------------------------------
# Input file
# -------------------------------
input_file = "results_exact.csv"

# -------------------------------
# Step 1: Robust manual parsing
# -------------------------------
rows = []

with open(input_file, "r") as f:
    for line_id, line in enumerate(f):
        line = line.strip()

        if not line:
            continue

        # Split ONLY first 3 commas
        parts = line.split(",", 3)

        if len(parts) < 4:
            print(f"Skipping malformed line {line_id}: {line}")
            continue

        file = parts[0]

        try:
            objective = float(parts[1])
            lb = float(parts[2])
        except ValueError:
            print(f"Skipping numeric parse error at line {line_id}")
            continue

        solution_raw = parts[3]
        
        if solution_raw == "[]":
            solution = []
            rows.append([file, objective, lb, solution])
            continue
        # -------------------------------
        # Fix solution parsing (robust)
        # -------------------------------
        try:
            solution_raw = solution_raw.strip()[1: -1]  # Remove outer quotes and brackets
            solution =  [int(x)+1 for x in solution_raw.split(",")]  # Convert to list of ints  

        except Exception:
            # try removing quotes if double-encoded
            try:
                solution_raw = solution_raw.strip('"').strip()[1: -1]  # Remove outer quotes and brackets
                solution =  [int(x)+1 for x in solution_raw.split(",")]  # Convert to list of ints  
                #print(solution)        
            except Exception:
                print(f"Warning: failed to parse solution at line {line_id}")
                solution = []

        rows.append([file, objective, lb, solution])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df_exact = pd.DataFrame(rows, columns=["file", "objective", "lb", "solution"])

# -------------------------------
# Step 3: Filename parser
# -------------------------------
def split_filename(fname):
    fname = fname.replace(".json", "")
    parts = fname.split("_")

    if len(parts) < 4:
        return pd.Series([fname, "", "", ""])

    if len(parts) > 4:
        instance_name = "_".join(parts[:-3])
        measure = parts[-3]
        percentile = parts[-2]
        algorithm = parts[-1]
    else:
        instance_name = parts[0]
        measure = parts[1]
        percentile = parts[2]
        algorithm = parts[3]

    return pd.Series([instance_name, measure, percentile, algorithm])

df_exact[["file_name", "measure", "percentile", "algorithm"]] = df_exact["file"].apply(split_filename)

# -------------------------------
# Step 4: Reorder columns
# -------------------------------
df_exact = df_exact[
    [
        "file_name",
        "measure",
        "percentile",
        "algorithm",
        "objective",
        "lb",
        "solution",
    ]
]

# -------------------------------
# Step 5: Derived metrics
# -------------------------------
df_exact["solution_size"] = df_exact["solution"].apply(len)

# Gap (important for exact solvers)
df_exact["gap"] = df_exact["lb"] - df_exact["objective"]

# -------------------------------
# Step 6: Memory optimization
# -------------------------------
df_exact["objective"] = df_exact["objective"].astype("float32")
df_exact["lb"] = df_exact["lb"].astype("float32")

# -------------------------------
# Step 7: Debug / preview
# -------------------------------

df_MIS_exact = df_exact[df_exact["algorithm"] == "alg0"]
df_MDS_exact = df_exact[df_exact["algorithm"] == "alg1"]

df_MDS_exact



Skipping numeric parse error at line 0


,file_name,measure,percentile,algorithm,objective,lb,solution,solution_size,gap
1,transoptas_5pso,correlation,p80,alg1,2.0,2.0,"[1201, 6565]",2,0.0
3,deepela_large_r1,cosine,p80,alg1,2.0,2.0,"[6576, 3251]",2,0.0
4,doe2vec,seuclidean,p90,alg1,2.0,2.0,"[1258, 2281]",2,0.0
5,doe2vec,correlation,p95,alg1,234.0,234.0,"[89, 208, 304, 395, 509, 511, 515, 518, 544, 5...",234,0.0
6,ela,seuclidean,p20,alg1,8280.0,8280.0,"[3676, 3677, 3678, 3679, 3680, 3681, 3682, 368...",8280,0.0
8,deepela_large_r1,correlation,p95,alg1,61.0,61.0,"[3577, 5998, 6337, 6710, 6807, 6576, 6605, 668...",61,0.0
9,ela,correlation,p90,alg1,5.0,5.0,"[1, 2535, 2878, 2883, 5040]",5,0.0
10,ela,seuclidean,p10,alg1,8280.0,8280.0,"[3676, 3677, 3678, 3679, 3680, 3681, 3682, 368...",8280,0.0
11,deepela_large_r1,correlation,p80,alg1,2.0,2.0,"[6576, 3254]",2,0.0
13,ela,correlation,p95,alg1,6.0,6.0,"[1, 1688, 2373, 2535, 2878, 5040]",6,0.0


In [6]:
import pandas as pd

# -------------------------------
# Step 1: Read CSV manually
# -------------------------------
input_file = "results_kmis_online_mis.csv"

data = []
with open(input_file, 'r') as f:
    header = f.readline()  # skip header
    for line in f:
        # split only on first and last comma
        file_part, rest = line.strip().split(',', 1)
        solution_part, objective_part = rest.rsplit(',', 1)
        # solution +1 to convert from 0-based to 1-based indexing
        if solution_part.strip()[1:-1].strip() == "...":
            solution_part = [x+1 for x in range(int(objective_part.strip()))]
        else:
            solution_part = [int(x.strip()) + 1 for x in solution_part.strip()[1:-1].strip().split(',')]
        
        data.append([file_part, solution_part, objective_part])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df = pd.DataFrame(data, columns=['file','solution','objective'])

# Convert objective to numeric
df['objective'] = pd.to_numeric(df['objective'], errors='coerce')

# -------------------------------
# Step 3: Parse filename
# -------------------------------
def parse_filename(fname):
    name = fname.replace(".out", "")
    parts = name.split("_")

    # Dataset name may have underscores
    dataset = "_".join(parts[:-3])
    measure = parts[-3]
    percentile = parts[-2]
    seed = parts[-1]

    return pd.Series([dataset, measure, percentile, seed])

df[['file_name','measure','percentile','seed']] = df['file'].apply(parse_filename)

# -------------------------------
# Step 4: Reorder columns
# -------------------------------
df = df[['file', 'file_name', 'measure', 'percentile', 'seed', 'solution', 'objective']]

# -------------------------------
# Step 5: Preview
# -------------------------------
df.head(30)

# -------------------------------
# Optional: Save cleaned CSV
# -------------------------------

## AGGREGATE:
df_online_kmis = df.drop(columns=["file"], axis=1)
df_online_kmis = df_online_kmis.sort_values(by=["file_name", "measure", "percentile", "seed"])
#df_online_kmis[df_online_kmis["file_name"] == "doe2vec"]

#tyni, cosine, p95
df_online_kmis #5pso representation is missing 


,file_name,measure,percentile,seed,solution,objective
0,deepela_large_r1,correlation,p10,20,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",505
1,deepela_large_r1,correlation,p10,30,"[46, 50, 52, 64, 92, 94, 95, 107, 109, 121, 12...",505
2,deepela_large_r1,correlation,p10,50,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",504
3,deepela_large_r1,correlation,p20,10,"[46, 47, 50, 52, 61, 64, 73, 92, 94, 95, 98, 1...",1055
4,deepela_large_r1,correlation,p20,20,"[46, 47, 50, 52, 61, 64, 73, 92, 94, 95, 98, 1...",1055
...,...,...,...,...,...,...
306,tinytla,cosine,p95,10,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7631
307,tinytla,cosine,p95,20,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7631
308,tinytla,cosine,p95,30,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7631
309,tinytla,cosine,p95,40,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7631


#REDU-MIS

In [7]:
import pandas as pd

# -------------------------------
# Step 1: Read CSV manually
# -------------------------------
input_file = "results_redumis.csv"

data = []
with open(input_file, 'r') as f:
    header = f.readline()  # skip header
    for line in f:
        # split only on first and last comma
        file_part, rest = line.strip().split(',', 1)
        solution_part, objective_part = rest.rsplit(',', 1)
        #print(solution_part)
        if solution_part.strip()  == "[]":
            solution_part = []
        elif solution_part.strip()[1:-1].strip() == "...":
            solution_part = [x + 1 for x in range(int(objective_part.strip()))]
        else:
            solution_part = [ int(x.strip()) + 1 for x in solution_part.strip()[1:-1].strip().split(",") ]

        data.append([file_part, solution_part, objective_part])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df = pd.DataFrame(data, columns=['file','solution','objective'])

# Convert objective to numeric
df['objective'] = pd.to_numeric(df['objective'], errors='coerce')

# -------------------------------
# Step 3: Parse filename
# -------------------------------
def parse_filename(fname):
    name = fname.replace(".out", "")
    parts = name.split("_")

    # Dataset name may have underscores
    dataset = "_".join(parts[:-5])
    measure = parts[-5]
    percentile = parts[-4]
    seed = parts[-3][4:]

    return pd.Series([dataset, measure, percentile, seed])

df[['file_name','measure','percentile','seed']] = df['file'].apply(parse_filename)

# -------------------------------
# Step 4: Reorder columns
# -------------------------------
df = df[['file', 'file_name', 'measure', 'percentile', 'seed', 'solution', 'objective']]

# -------------------------------
# Step 5: Preview
# -------------------------------
df.head(30)

# -------------------------------
# Optional: Save cleaned CSV
# -------------------------------

## AGGREGATE:
df_redumis = df.drop(columns=["file"], axis=1).copy()

df_redumis

,file_name,measure,percentile,seed,solution,objective
0,deepela_large_r1,correlation,p10,30,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",505
1,deepela_large_r1,correlation,p10,40,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",505
2,deepela_large_r1,correlation,p10,50,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",505
3,deepela_large_r1,correlation,p20,20,"[46, 47, 50, 52, 61, 64, 73, 92, 94, 95, 98, 1...",1055
4,deepela_large_r1,correlation,p20,50,"[46, 47, 50, 52, 61, 64, 73, 92, 94, 95, 98, 1...",1055
...,...,...,...,...,...,...
310,transoptas_5pso,seuclidean,p90,50,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",6087
311,transoptas_5pso,seuclidean,p95,20,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",6790
312,transoptas_5pso,seuclidean,p95,30,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",6790
313,transoptas_5pso,seuclidean,p95,40,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",6790


#NUMDS

In [8]:
import pandas as pd
import ast  # for safely converting solution string to list

# -------------------------------
# Step 1: Read CSV safely
# -------------------------------
input_file = "results_numds.csv"

data = []
with open(input_file, 'r') as f:
    header = f.readline()  # skip header
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # Split only on the first three commas
        first_comma = line.find(',')
        second_comma = line.find(',', first_comma + 1)
        third_comma = line.find(',', second_comma + 1)
        
        name_part = line[:first_comma]
        seed_part = line[first_comma+1:second_comma]
        obj_part = line[second_comma+1:third_comma]
        sol_part = line[third_comma+1:]  # rest of line is solution
        
        # Convert numeric fields
        try:
            seed_val = int(seed_part)
            obj_val = int(obj_part)
        except ValueError:
            print(f"Skipping line (cannot parse numbers): {line}")
            continue
        
        data.append([name_part, seed_val, obj_val, sol_part])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df = pd.DataFrame(data, columns=['name','seed','objective','solution'])

# -------------------------------
# Step 3: Parse 'name' into dataset, measure, percentile
# -------------------------------
def parse_name(name):
    parts = name.split("_")
    if len(parts) < 3:
        return pd.Series([name, "", ""])
    dataset = "_".join(parts[:-2])
    measure = parts[-2]
    percentile = parts[-1]
    return pd.Series([dataset, measure, percentile])

df[['file_name','measure','percentile']] = df['name'].apply(parse_name)

# -------------------------------
# Step 4: Optional: Convert solution string to Python list
# -------------------------------
def parse_solution(sol_str):
    try:
        # safely convert string '[1,2,3]' to list
        return ast.literal_eval(sol_str)
    except:
        return []

df['solution_list'] = df['solution'].apply(parse_solution)

# -------------------------------
# Step 5: Reorder columns
# -------------------------------
df = df[['name','file_name','measure','percentile','seed','solution','objective']]

# -------------------------------
# Step 6: Preview
# --------------------------------

df_numds = df 
df_numds.drop(columns=["name"], axis=1, inplace=True)
df_numds = df_numds.sort_values(by=["file_name", "measure", "percentile", "seed"])

df_numds 
 

,file_name,measure,percentile,seed,solution,objective
72,deepela_large_r1,correlation,p10,10,[4359],1
74,deepela_large_r1,correlation,p10,20,[4359],1
129,deepela_large_r1,correlation,p10,30,[4359],1
319,deepela_large_r1,correlation,p10,40,[4359],1
194,deepela_large_r1,correlation,p10,50,[4359],1
...,...,...,...,...,...,...
249,transoptas_5pso,seuclidean,p95,10,"[556, 4335]",2
206,transoptas_5pso,seuclidean,p95,20,"[556, 4335]",2
335,transoptas_5pso,seuclidean,p95,30,"[556, 4335]",2
161,transoptas_5pso,seuclidean,p95,40,"[556, 4335]",2


In [9]:
#PREPARATION: df_greedy, df_exact, df_online_kmis, df_redumis, df_numds

df_approx = df_greedy[["file_name", "measure", "percentile", "algorithm", "solutions", "objective" ]]
df_approx_mis = df_approx[df_approx["algorithm"] == "alg0"]
df_approx_mds = df_approx[df_approx["algorithm"] == "alg1"]
#renaming

df_approx_mis.rename(columns={"objective" : "Greedy"}, inplace=True)
df_approx_mds.rename(columns={"objective" : "Greedy"}, inplace=True)
df_approx_mis.rename(columns={"solutions" : "solution_greedy"}, inplace=True)
df_approx_mds.rename(columns={"solutions" : "solution_greedy"}, inplace=True)

#df_approx_mis
#### EXACT SOLVERS

df_exact = df_exact[["file_name", "measure", "percentile", "algorithm", "objective", "lb", "solution" ]]
df_exact_mis = df_exact[df_exact["algorithm"] == "alg0"]
df_exact_mds = df_exact[df_exact["algorithm"] == "alg1"]

df_exact_mis.rename(columns={"objective" : "CPLEX"}, inplace=True) 
df_exact_mds.rename(columns={"objective" : "CPLEX"}, inplace=True) 
df_exact_mis.rename(columns={"solution" : "solution_exact"}, inplace=True) 
df_exact_mds.rename(columns={"solution" : "solution_exact"}, inplace=True)

df_approx_mis.drop(columns=["algorithm"], axis=1, inplace=True)
df_exact_mis.drop(columns=["algorithm"], axis=1, inplace=True)

df_approx_mds.drop(columns=["algorithm"], axis=1, inplace=True)
df_exact_mds.drop(columns=["algorithm"], axis=1, inplace=True)

### KMIS online:

#df_nclue = df_online_kmis[["instance_name", "measure", "percentile", "seed",  "objective" ]]
#df_nclue
df_online_kmis
df_online_kmis.rename(columns={"objective" : "objective_kmis"}, inplace=True)
df_online_kmis.rename(columns={"solution" : "solution_kmis"}, inplace=True)

df_online_kmis


/ceph/hpc/home/djukanovicm/.local/lib/python3.6/site-packages/pandas/core/frame.py:4308: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,
/ceph/hpc/home/djukanovicm/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


,file_name,measure,percentile,seed,solution_kmis,objective_kmis
0,deepela_large_r1,correlation,p10,20,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",505
1,deepela_large_r1,correlation,p10,30,"[46, 50, 52, 64, 92, 94, 95, 107, 109, 121, 12...",505
2,deepela_large_r1,correlation,p10,50,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",504
3,deepela_large_r1,correlation,p20,10,"[46, 47, 50, 52, 61, 64, 73, 92, 94, 95, 98, 1...",1055
4,deepela_large_r1,correlation,p20,20,"[46, 47, 50, 52, 61, 64, 73, 92, 94, 95, 98, 1...",1055
...,...,...,...,...,...,...
306,tinytla,cosine,p95,10,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7631
307,tinytla,cosine,p95,20,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7631
308,tinytla,cosine,p95,30,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7631
309,tinytla,cosine,p95,40,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7631


In [128]:
## results_redumis.csv:
df_redumis = df_redumis[["file_name", "measure", "percentile", "seed", "solution", "objective" ]]
df_redumis

# Group by instance_name, measure, percentile, and compute mean objective
#df_redumis_grouped = df_redumis.groupby(["file_name", "measure", "percentile"], as_index=False).agg({
#    "objective": "mean"
#})

df_redumis.rename(columns={"objective" : "objective_redumis"}, inplace=True)
df_redumis.rename(columns={"solution" : "solution_redumis"}, inplace=True)

#df_redumis_grouped.drop(columns=["seed"], axis=1, inplace=True)
# Preview
#df_redumis_grouped.head(15)
df_redumis

,file_name,measure,percentile,seed,solution_redumis,objective_redumis
0,deepela_large_r1,correlation,p10,30,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",505
1,deepela_large_r1,correlation,p10,40,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",505
2,deepela_large_r1,correlation,p10,50,"[46, 50, 52, 64, 94, 95, 109, 121, 122, 124, 1...",505
3,deepela_large_r1,correlation,p20,20,"[46, 47, 50, 52, 61, 64, 73, 92, 94, 95, 98, 1...",1055
4,deepela_large_r1,correlation,p20,50,"[46, 47, 50, 52, 61, 64, 73, 92, 94, 95, 98, 1...",1055
...,...,...,...,...,...,...
310,transoptas_5pso,seuclidean,p90,50,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",6087
311,transoptas_5pso,seuclidean,p95,20,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",6790
312,transoptas_5pso,seuclidean,p95,30,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",6790
313,transoptas_5pso,seuclidean,p95,40,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",6790


In [8]:
## numds
df_numds
df_numds = df_numds[["file_name", "measure", "percentile", "seed",  "objective", "solution" ]]

df_numds.rename(columns={"solution" : "solution_numds" },   inplace=True)

In [10]:
## TO MERGE (MIS solvers):  

print("MIS solvers")
# List of dataframes to merge

def merge_and_output(dfs_to_merge = [df_online_kmis, df_redumis], columns_to_join = ["file_name", "measure", "percentile", "seed"], \
                     out_name="MIS_2mh_solvers_results.csv"):


    # Start with the first dataframe
    df_merged = dfs_to_merge[0]

    # Merge sequentially on the key columns
    for df in dfs_to_merge[1: ]:
        df_merged = df_merged.merge(df, on=columns_to_join, how="left")

    # Preview the merged dataframe  
    
    df_merged.to_csv(out_name)
    return df_merged

df_merged = merge_and_output()


MIS solvers


In [39]:
### work with datasets: MIS: df_online_kmis, df_redumis,  df_approx_mis, df_exact_mis, 
## MDS: df_numds, df_approx_mds, df_exact_mds
# 

df_approx_mds



,file_name,measure,percentile,solution_greedy,Greedy
1,transoptas_5pso,correlation,p80,"[6565, 4326, 6566, 6567, 4398]",5
5,deepela_large_r1,cosine,p80,"[6798, 6860, 6804, 6807, 4503, 6810, 4515, 656...",64
6,transoptas_5pso,correlation,p20,[175],1
8,doe2vec,seuclidean,p90,"[4354, 2281, 1258, 1259, 5753, 5754]",6
10,doe2vec,correlation,p20,"[5012, 4245, 4191]",3
...,...,...,...,...,...
168,transoptas_5pso,seuclidean,p50,"[4710, 4335]",2
170,deepela_large_r1,seuclidean,p10,[3755],1
172,tinytla,correlation,p10,[3139],1
175,ela,cosine,p80,"[6753, 2342, 7465, 4938, 4940, 1549, 5071, 424...",13


In [ ]:
### work with datasets: MIS: df_online_kmis, df_redumis,  df_approx_mis, df_exact_mis, 
## MDS: df_numds, df_approx_mds, df_exact_mds
# 

### save MIS:

df_online_kmis.to_csv("MIS_kmis_online_all_nodes.csv", index=False)
df_redumis.to_csv("MIS_redumis_results_all_nodes.csv", index=False)  
df_approx_mis.to_csv("MIS_greedy_results_all_nodes.csv", index=False)
df_exact_mis.to_csv("MIS_exact_results_all_nodes.csv", index=False)

### save MDS:
df_numds.to_csv("MDS_numds_results_all_nodes.csv", index=False )
df_approx_mds.to_csv("MDS_greedy_results_all_nodes.csv", index=False)
df_exact_mds.to_csv("MDS_exact_results_all_nodes.csv", index=False)




## SPLIT (70:30) RESULTS 


### read EXACT APPROACH FOR MIS AND MDS 


In [ ]:
# ------------------ LOAD CSV ------------------
df = pd.read_csv("results_deterministic_train_test_alg01.csv")

# ------------------ PARSE SOLUTION ------------------
# Convert string "[1,2,3]" → list
df["solution"] = df["solution"].apply(
    lambda x: str([i + 1 for i in ast.literal_eval(x)])
)
# ------------------ PARSE FILE NAME ------------------
def parse_filename(fname):
    # remove extension
    fname = fname.replace(".out", "")

    parts = fname.split("_")

    # last parts are always: percentile, alg, train/test (ela_correlation_p50_train_test_nodes_split_2_alg1_test.out)
    split_id = parts[-3] 
    percentile = parts[-8]
    algorithm = parts[-2]
    split = parts[-1]

    # measure is just before percentile
    measure = parts[-9]

    # everything before that is dataset name
    dataset = "_".join(parts[: (len(parts)-9)])   #parts[:-10])

    return pd.Series([dataset, measure, percentile, algorithm, split, split_id, ])


df[["dataset", "measure", "percentile", "algorithm", "split", "split_id"]] = df["file"].apply(parse_filename)

# ------------------ KEEP ONLY IMPORTANT COLUMNS ------------------
df = df[[
    "dataset",
    "measure",
    "percentile",
    "algorithm",
    "split",
    "split_id",
    "objective",
    "LB",
    "solution"
]]

# ------------------ DONE ------------------
df_MIS_exact=df[df["algorithm"] == "alg0"]
df_MDS_exact=df[df["algorithm"] == "alg1"]

df_MIS_exact.drop(columns=["algorithm"], axis=1, inplace=True)
df_MDS_exact.drop(columns=["algorithm"], axis=1, inplace=True)

df_MDS_exact


/ceph/hpc/home/djukanovicm/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


,dataset,measure,percentile,split,split_id,objective,LB,solution
0,ela,cosine,p50,train,2,1.0,1,[2535]
2,ela,correlation,p50,test,2,1.0,1,[2878]
3,deepela_large_r1,cosine,p50,test,4,2.0,2,"[1001, 6576]"
4,tinytla,cosine,p95,test,5,55.0,55,"[3, 18, 110, 712, 817, 844, 856, 955, 1390, 13..."
5,transoptas_5pso,correlation,p80,train,1,2.0,2,"[1201, 6565]"
...,...,...,...,...,...,...,...,...
727,tinytla,correlation,p50,test,4,2.0,2,"[822, 1108]"
728,deepela_large_r1,seuclidean,p50,test,2,1.0,1,[1058]
729,doe2vec,seuclidean,p50,train,1,2.0,2,"[1003, 2281]"
730,doe2vec,cosine,p90,train,5,38.0,38,"[88, 208, 380, 1104, 1202, 2266, 2276, 2281, 2..."


In [40]:
# ------------------ LOAD CSV ------------------
df = pd.read_csv("results_deterministic_train_test_alg23.csv")

# ------------------ PARSE SOLUTION ------------------
# Convert string "[1,2,3]" → list
import ast

df["solution"] = df["solution"].apply(
    lambda x: str([i + 1 for i in ast.literal_eval(x)])
)

# ------------------ PARSE FILE NAME ------------------
def parse_filename(fname):
    # remove extension
    fname = fname.replace(".out", "")

    parts = fname.split("_")

    # last parts are always: percentile, alg, train/test (ela_correlation_p50_train_test_nodes_split_2_alg1_test.out)
    split_id = parts[-3] 
    percentile = parts[-8]
    algorithm = parts[-2]
    split = parts[-1]

    # measure is just before percentile
    measure = parts[-9]

    # everything before that is dataset name
    dataset = "_".join(parts[: (len(parts)-9)])   #parts[:-10])

    return pd.Series([dataset, measure, percentile, algorithm, split, split_id, ])


df[["dataset", "measure", "percentile", "algorithm", "split", "split_id"]] = df["file"].apply(parse_filename)

# ------------------ KEEP ONLY IMPORTANT COLUMNS ------------------
df = df[[
    "dataset",
    "measure",
    "percentile",
    "algorithm",
    "split",
    "split_id",
    "objective",
    "solution"
]]

# ------------------ DONE ------------------
df_MIS_greedy=df[df["algorithm"] == "alg2"]
df_MDS_greedy=df[df["algorithm"] == "alg3"]

df_MDS_greedy


,dataset,measure,percentile,algorithm,split,split_id,objective,solution
2,transoptas_5pso,cosine,p90,alg3,train,1,331,"[6657, 6659, 6660, 6718, 6719, 6666, 1035, 667..."
3,deepela_large_r1,seuclidean,p50,alg3,test,3,1,[3469]
6,tinytla,cosine,p90,alg3,train,2,321,"[2049, 1026, 1028, 2565, 1031, 1032, 1034, 103..."
7,deepela_large_r1,correlation,p90,alg3,train,1,92,"[6666, 526, 4641, 6706, 6707, 6708, 6709, 6710..."
8,deepela_large_r1,cosine,p90,alg3,test,1,199,"[6663, 524, 16, 529, 1059, 4650, 4653, 4662, 6..."
...,...,...,...,...,...,...,...,...
1511,transoptas_5pso,cosine,p95,alg3,train,1,676,"[25, 26, 27, 42, 4146, 2101, 56, 4155, 4158, 2..."
1517,tinytla,cosine,p95,alg3,train,3,368,"[2049, 16, 17, 22, 23, 26, 27, 28, 2101, 2102,..."
1524,transoptas_5pso,correlation,p20,alg3,train,5,1,[190]
1528,tinytla,seuclidean,p80,alg3,test,4,1,[45]


## kmis-online

In [ ]:
# ------------------ PARSE SOLUTION -----------------

import pandas as pd
import ast

input_path = "results_kmis_online_mis_5train_test.csv"


# ------------------ PARSE FILENAME ------------------
def parse_filename(fname):


    fname = fname.replace(".out", "")
    parts = fname.split("_")

    metric_idx = next(i for i, p in enumerate(parts)
                      if p in ["correlation", "seuclidean", "cosine"])
    representation = "_".join(parts[:metric_idx])

    measure = parts[metric_idx]

    percentile = next(p for p in parts if p.startswith("p"))

    split_id = None
    if "split" in parts:
        split_idx = parts.index("split")
        split_id = int(parts[split_idx + 1])

    mode = None
    for p in reversed(parts):
        if p in ["train", "test"]:
            mode = p
            break

    alg = None
    for p in parts:
        if p.startswith("alg"):
            alg = int(p.replace("alg", ""))

    return representation, measure, percentile, split_id, mode, alg


# ------------------ FAST SOLUTION PARSER ------------------
def fast_parse_solution(sol_str):
    sol_str = sol_str.strip()

    # remove brackets safely
    if sol_str.startswith("["):
        sol_str = sol_str[1:]
    if sol_str.endswith("]"):
        sol_str = sol_str[:-1]

    if sol_str == "":
        return []

    return [int(x.strip()) for x in sol_str.split(",") if x.strip()]


# ------------------ PARSE LINE ------------------
def parse_line(line):
    parts = line.split(",", 4)

    fname, rest = line.split(",", 1)
    fname = fname.strip()

    # extract solution safely (everything between first '[' and last ']')
    start = rest.find("[")
    end = rest.rfind("]")

    if start == -1 or end == -1:
        return fname, None, None, 0, []

    sol_str = rest[start:end+1]

    try:
        solution = [i + 1 for i in ast.literal_eval(sol_str)]
    except Exception:
        solution = fast_parse_solution(sol_str)

    objective = len(solution)

    return fname, None, None, objective, solution


# ------------------ MAIN ------------------
def main(input_path):
    rows = []

    with open(input_path, "r") as f:
        for line in f:
            line = line.strip()

            # skip header / empty
            if not line or line.startswith("name"):
                continue

            # skip clearly broken lines
            if line.count("[") != line.count("]"):
                print(f"[WARNING] Skipping broken line:\n{line[:120]}...")
                continue

            try:
                fname, seed, time, objective, solution = parse_line(line)
                #print(f"Parsed line: {fname}, objective: {objective}, solution size: {len(solution)}")  # Debug

                representation, measure, percentile, split_id, mode, alg = parse_filename(fname)

                #print(f"Parsed: {fname} → {representation}, {measure}, {percentile}, split {split_id}, mode {mode}, alg {alg}")

                rows.append({
                    "file": fname,
                    "representation": representation,   # ✅ fixed typo
                    "measure": measure,
                    "percentile": percentile,
                    "split_id": split_id,
                    "mode": mode,
                    #"alg": alg,
                    #"seed": seed,
                    #"time": time,
                    "objective": objective,
                    #"solution_size": len(solution),
                    "solution": solution
                })

            except Exception as e:
                print(f"[ERROR] Failed parsing line:\n{line[:120]}...\n{e}")

    df = pd.DataFrame(rows)
    return df


# ------------------ RUN ------------------
df_kmis_online = main(input_path)
df_kmis_online 



Parsed line: doe2vec_correlation_p10_train_test_nodes_split_1_test_seed10_t600_redumis.out, objective: 172, solution size: 172
Parsed line: doe2vec_correlation_p10_train_test_nodes_split_1_test_seed20_t600_redumis.out, objective: 172, solution size: 172
Parsed line: doe2vec_correlation_p10_train_test_nodes_split_1_test_seed30_t600_redumis.out, objective: 172, solution size: 172
Parsed line: doe2vec_correlation_p10_train_test_nodes_split_1_test_seed40_t600_redumis.out, objective: 172, solution size: 172
Parsed line: doe2vec_correlation_p10_train_test_nodes_split_1_test_seed50_t600_redumis.out, objective: 172, solution size: 172
Parsed line: doe2vec_correlation_p10_train_test_nodes_split_1_train_seed10_t600_redumis.out, objective: 403, solution size: 403
Parsed line: doe2vec_correlation_p10_train_test_nodes_split_1_train_seed20_t600_redumis.out, objective: 403, solution size: 403
Parsed line: doe2vec_correlation_p10_train_test_nodes_split_1_train_seed30_t600_redumis.out, objective: 403, 

,file,representation,measure,percentile,split_id,mode,objective,solution
0,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172,"[22, 30, 37, 54, 59, 60, 66, 73, 82, 83, 84, 8..."
1,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172,"[22, 30, 37, 54, 59, 60, 65, 66, 73, 82, 83, 8..."
2,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172,"[22, 30, 37, 54, 59, 60, 66, 73, 82, 83, 84, 8..."
3,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172,"[22, 30, 37, 54, 59, 60, 66, 73, 82, 83, 84, 8..."
4,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172,"[22, 30, 37, 54, 59, 60, 66, 73, 82, 83, 84, 8..."
...,...,...,...,...,...,...,...,...
2939,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,5,train,4709,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15..."
2940,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,5,train,4709,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15..."
2941,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,5,train,4709,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15..."
2942,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,5,train,4709,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15..."


In [23]:
df_kmis_small = df_kmis_online.drop(columns=["solution"])
df_kmis_small

,file,representation,measure,percentile,split_id,mode,objective
0,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172
1,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172
2,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172
3,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172
4,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,1,test,172
...,...,...,...,...,...,...,...
2939,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,5,train,4709
2940,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,5,train,4709
2941,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,5,train,4709
2942,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,5,train,4709


## REDUMIS (results_redumis_5train_test.csv): TODO: convert node indices from solution (.txt file to be used)

In [30]:
input_path = "results_redumis_5train_test.csv"

import os
import ast
import pandas as pd
import argparse



# ------------------ PARSE FILENAME ------------------
def parse_filename(fname):
    fname = fname.replace(".out", "")
    parts = fname.split("_")

    # ---- dataset (everything before metric) ----
    metric_idx = next(i for i, p in enumerate(parts) if p in ["correlation", "seuclidean", "cosine"])
    dataset = "_".join(parts[:metric_idx])

    # ---- measure ----
    measure = parts[metric_idx]

    # ---- percentile ----
    percentile = next(p for p in parts if p.startswith("p"))

    # ---- split_id ----
    split_id = None
    if "split" in parts:
        split_idx = parts.index("split")
        split_id = parts[split_idx + 1]

    # ---- train/test (last occurrence matters!) ----
    split = None
    for p in reversed(parts):
        if p in ["train", "test"]:
            split = p
            break

    # ---- seed ----
    seed = None
    for p in parts:
        if p.startswith("seed"):
            seed = int(p.replace("seed", ""))

    # ---- algorithm (last token, unless it's tXXX etc.) ----
    algorithm = parts[-1]

    return dataset, measure, percentile, split, split_id, seed, algorithm




# ------------------ PARSE LINE ------------------
def parse_line(line):
    # split only first and last comma safely
    first_comma = line.find(",")
    last_comma = line.rfind(",")

    fname = line[:first_comma].strip()
    solution_str = line[first_comma + 1:last_comma].strip()
    solution_str = str([  int(x.strip())+1 for x in solution_str.strip()[1:-1].split(",") if x.strip() ] ) # safely convert to list of strings  
    objective = int(line[last_comma + 1:].strip())

    solution = solution_str #ast.literal_eval(solution_str)

    return fname, solution, objective


# ------------------ MAIN ------------------
def main(input_path):

    rows = []

    with open(input_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                fname, solution, objective = parse_line(line)

                dataset, measure, percentile, split, split_id, seed, algorithm = parse_filename(fname)

                rows.append({
                    "file": fname,
                    "dataset": dataset,
                    "measure": measure,
                    "percentile": percentile,
                    "split": split,
                    "split_id": split_id,
                    "seed": seed,
                    "algorithm": algorithm,
                    "objective": objective,
                    "solution_size": len(solution),
                    "solution": solution
                })

            except Exception as e:
                print(f"[ERROR] Failed parsing line:\n{line}\n{e}")

    df = pd.DataFrame(rows)
    return df
    # save
    #df.to_csv(args.output_csv, index=False)

    #print(f"\nSaved to: {args.output_csv}")


if __name__ == "__main__":
    df_redumis = main(input_path)

    

In [41]:
#df_redumis.drop(columns=["file"], axis=1, inplace=True)

df_redumis = df_redumis.sort_values(by=["dataset", "measure", "percentile", "split", "split_id", "seed"])


In [ ]:
## kmis-online 

,file,dataset,measure,percentile,split,split_id,seed,algorithm,objective,solution_size,solution
0,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,10,redumis,172,966,"[22, 30, 37, 54, 59, 60, 66, 73, 82, 83, 84, 8..."
1,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,20,redumis,172,963,"[22, 30, 37, 54, 59, 60, 65, 66, 73, 82, 83, 8..."
2,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,30,redumis,172,967,"[22, 30, 37, 54, 59, 60, 66, 73, 82, 83, 84, 8..."
3,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,40,redumis,172,964,"[22, 30, 37, 54, 59, 60, 66, 73, 82, 83, 84, 8..."
4,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,50,redumis,172,965,"[22, 30, 37, 54, 59, 60, 66, 73, 82, 83, 84, 8..."
...,...,...,...,...,...,...,...,...,...,...,...
2939,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,train,5,10,redumis,4709,27379,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15..."
2940,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,train,5,20,redumis,4709,27379,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15..."
2941,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,train,5,30,redumis,4709,27379,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15..."
2942,transoptas_5pso_seuclidean_p95_train_test_node...,transoptas_5pso,seuclidean,p95,train,5,40,redumis,4709,27379,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15..."


## NuMDS

In [42]:
input_path = "results_numds_5train_test.csv"

import pandas as pd
import ast

rows = []

with open(input_path, "r") as f:
    next(f)  # skip header
    
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # Fix CSV issue (list column)
        parts = line.split(",", 3)
        
        name = parts[0]
        seed = int(parts[1])
        obj = int(parts[2])
        sol = ast.literal_eval(parts[3])
        
        # ---- Parse name ----
        tokens = name.split("_")
        
        # Find key positions
        metric_idx = next(i for i, t in enumerate(tokens) if t in ["correlation", "seuclidean", "cosine"])
        p_idx = next(i for i, t in enumerate(tokens) if t.startswith("p"))
        split_idx = tokens.index("split")
        
        representation = "_".join(tokens[:metric_idx])
        metric = tokens[metric_idx]
        percentile = tokens[p_idx]
        nodes_split = tokens[split_idx + 1]# int(tokens[split_idx + 1])
        mode = tokens[-1]  # train/test
        
        rows.append({
            "name": name,
            "representation": representation,
            "metric": metric,
            "percentile": percentile,
            "nodes_split": nodes_split,
            "mode": mode,
            "seed": seed,
            "obj": obj,
            "sol": sol
        })

df = pd.DataFrame(rows)
df = df.sort_values(by=["representation", "metric", "percentile", "nodes_split", "mode", "seed"])
df[ (df["representation"] == "doe2vec" ) &  (df["metric"] == "correlation") & (df["percentile"] == "p50")  ]

df_numds=df.copy()
df_numds.drop(columns=["name"], axis=1, inplace=True)

df_numds.sort_values(by=["representation", "metric", "percentile", "nodes_split", "seed", "mode"], inplace=True)

#df_approx_mds.sort_values(by=["file_name", "measure", "percentile"], inplace=True)
#df_approx_mds.head(20)

In [ ]:
#MIS: df_MIS_greedy, df_MIS_exact, df_kmis_online, df_redumis
#MDS: df_numds, df_MDS_greedy, df_MDS_exact


#df_MIS_greedy.rename({"dataset": "representation"}, inplace=True, axis=1)
#df_MIS_exact.rename({"dataset": "representation"}, inplace=True, axis=1)
#df_redumis.rename({"dataset": "representation"}, inplace=True, axis=1)
#df_kmis_online.drop(columns=["file"], axis=1, inplace=True)

#df_redumis.drop(columns=["file"], axis=1, inplace=True)
#df_redumis.rename(columns={"dataset": "representation"}, inplace=True)

#df_MDS_greedy.rename(columns={"dataset": "representation"}, inplace=True)
#df_MDS_exact.rename(columns={"dataset": "representation"}, inplace=True)    
#df_redumis.drop(columns=["algorithm"], axis=1, inplace=True)

#df_MIS_exact.rename({"dataset": "representation"}, inplace=True, axis=1)
#df_numds.rename(columns={"nodes_split": "split_id"}, inplace=True)

#convert type df_kmis_online of "split_id" into str
#df_kmis_online["split_id"] = df_kmis_online["split_id"].astype(str)


In [98]:
MIS_dfs ={ "MIS_kmis_online_split" : df_kmis_online, "MIS_redumis_split" : df_redumis, "MIS_greedy_split" : df_MIS_greedy, "MIS_exact_split" : df_MIS_exact }
MDS_dfs = { "MDS_numds_split" : df_numds, "MDS_greedy_split" : df_MDS_greedy, "MDS_exact_split" : df_MDS_exact }

# filter now and save to specific csvs according to split_id
for split_id in range(1, 6):
    for title, df in MIS_dfs.items():

        if "split_id" not in df.columns:
            print(f"[WARNING] {title} missing split_id")
            continue

        df_train = df[df["split_id"] == str(split_id)]
        df_train.to_csv(f"{title}_train_split_{split_id}.csv", index=False)
        
    for title, df in MDS_dfs.items():
        
        if "split_id" not in df.columns:
            print(f"[WARNING] {title} missing split_id")
            continue
        df_train = df[df["split_id"] == str(split_id)]
        df_train.to_csv(f"{title}_train_split_{split_id}.csv", index=False)


df_numds.info()

        

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2645 entries, 1672 to 2487
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   representation  2645 non-null   object
 1   metric          2645 non-null   object
 2   percentile      2645 non-null   object
 3   split_id        2645 non-null   object
 4   mode            2645 non-null   object
 5   seed            2645 non-null   int64 
 6   obj             2645 non-null   int64 
 7   sol             2645 non-null   object
dtypes: int64(2), object(6)
memory usage: 186.0+ KB


In [96]:
df_kmis_online.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2944 entries, 0 to 2943
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   representation  2944 non-null   object
 1   measure         2944 non-null   object
 2   percentile      2944 non-null   object
 3   split_id        2944 non-null   int64 
 4   mode            2944 non-null   object
 5   objective       2944 non-null   int64 
 6   solution        2944 non-null   object
dtypes: int64(2), object(5)
memory usage: 161.1+ KB
